## Evaluating RAG

Using scientific rigor to evaluate an artform.

We can either evaluate retrieval performance or the answer it self.

### RAG Pros 
* Quick
* Enables expertise
* Saves context length and focuses it on what is relevant

### RAG Cons
* Very empirical -> Trial & Error -> lots of Models and techniques (chunking, retrieval methods)
* The unexpected I dont know -> hard to understand where things go wrong when they do


### Steps

1. Curate Test Set - Example question and answer set with the right context identified and relevance in response (Our Benchmark)

i.e., Use real world conversations from past customer service interactions. Test should also push the model to generalise well

2. Measure retrival (How good can we surface context/performance) - Mean Reciprocal Rank (MRR), Normalised Discounted Cumulative Gain (nDCG), Recall@K & Precision@K

3. Measure Answers (End user objective & Feedback) - LLM-as-aJudge to score answers against provided criteria (Accuracy, Relevance, Completeness) and rubric

### Definitions

**MRR** - Average inverse rank of the first hit; 1 if first chunk always has relevant context, 1/2 if 2nd, 1/3 if third chunk. Take the average of all hits in the test set

**nDCG** - Looks at all hits rather than the first. How well distributed are the hits in the chunks. Are the top few ones relevant - 1 if so.

**Recall@K** (More important) - Proportion of tests where relevant context was in the top K chunks. If we have multiple key words we can use **Keyword Coverage** as a similar metric.

**Precision@K** - Proportion of Top K chunks that are relevant


### Experimentation

Experiment with how these metrics vary based on chunk size and different Models


In [1]:
# eval folder -> JSONL = file where each line is a JSON Doc/Object -> very convenient to append to like a JSON CSV
from evaluation import test

tests= test.load_tests()
len(tests)

150

In [2]:
# test.py use pydantic objects

#load first test question
question1 = tests[0]
print(question1.category)
print(question1.question)
print(question1.keywords)
print(question1.reference_answer)

direct_fact
Who won the prestigious IIOTY award in 2023?
['Maxine', 'Thompson', 'IIOTY']
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.


In [3]:
from collections import Counter
count = Counter((t.category for t in tests))
count

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [4]:
from evaluation.eval import evaluate_retrieval, evaluate_answer

In [ ]:
# double check embedding model is consistent, restart kernel after ingestion
evaluate_retrieval(question1)

RetrievalEval(mrr=0.16666666666666666, ndcg=0.28711770538226206, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

In [7]:
# mathc 3 outputs to functions 3 outputs
eval, answer, chunks = evaluate_answer(question1)
eval

AnswerEval(feedback="The answer correctly identifies Maxine as the recipient of the IIOTY award in 2023 and mentions her specific contribution. It also correctly names the award as the Insurellm IIOTY Innovator Award, aligning with the reference. However, it uses the first name only ('Maxine'), whereas the reference specifies the full name ('Maxine Thompson'), which is a minor but notable omission. The answer is concise and directly relevant, but it could be slightly more complete by including her full name.", accuracy=5.0, completeness=4.0, relevance=5.0)

In [9]:
print(eval.accuracy)
print(eval.relevance)
print(eval.completeness)
print(eval.feedback)


5.0
5.0
4.0
The answer correctly identifies Maxine as the recipient of the IIOTY award in 2023 and mentions her specific contribution. It also correctly names the award as the Insurellm IIOTY Innovator Award, aligning with the reference. However, it uses the first name only ('Maxine'), whereas the reference specifies the full name ('Maxine Thompson'), which is a minor but notable omission. The answer is concise and directly relevant, but it could be slightly more complete by including her full name.
